## Imports and loading data from task 1

In [1]:
import os
import numpy as np
import pandas as pd
import wfdb

PATH_RAW = "../data/1_raw/mapped_beats_df.csv"
PATH_OUTPUT = "../data/2_extracted/features_extracted.csv"

print("Loading data of mapped beats...")
mapped_beats_df = pd.read_csv(PATH_RAW)

# Obtain unique records present in the data
records_to_process = mapped_beats_df["record_id"].unique()
print(f"Records to process: {list(records_to_process)}")

Loading data of mapped beats...
Records to process: [np.int64(100), np.int64(101), np.int64(105), np.int64(106), np.int64(108), np.int64(109), np.int64(111), np.int64(112), np.int64(115), np.int64(117), np.int64(119), np.int64(201), np.int64(213), np.int64(219)]


## Feature extraction function

In [3]:
def extract_features_for_record(rec_id, df_rec_beats):
    """
    Download the signal from PhysioNet and compute the 8 physical 
    features for each valid heartbeat in a specific record.
    """
    print(f"-> Processing record {rec_id}...")
    
    # Download/read the original record directly from PhysioNet.
    record = wfdb.rdrecord(str(rec_id), pn_dir="mitdb")
    signal = record.p_signal[:, 0]  # Channel 0 (MLII lead)
    fs = record.fs                  # Sampling frequency (360 Hz)
    
    # Extract the sample indices (R-peaks)
    r_peaks = df_rec_beats["sample"].values
    n_beats = len(r_peaks)
    
    # --- A. Temporal features: RR intervals (in ms) ---
    # Difference in samples between consecutive R-peaks
    rr_samples = np.diff(r_peaks)
    rr_ms_all = (rr_samples / fs) * 1000.0  # to miliseconds
    
    rr_current = np.zeros(n_beats)
    rr_prev = np.zeros(n_beats)
    rr_local_mean = np.zeros(n_beats)
    rr_ratio = np.zeros(n_beats)
    
    # Handling the "First beat edge case"
    # The mean of the first 5 available intervals in the record is used
    initial_mean_rr = np.mean(rr_ms_all[:5]) if len(rr_ms_all) >= 5 else np.mean(rr_ms_all)
    
    for i in range(n_beats):
        if i == 0:
            rr_current[i] = initial_mean_rr
            rr_prev[i] = initial_mean_rr
            rr_local_mean[i] = initial_mean_rr
        elif i == 1:
            rr_current[i] = rr_ms_all[0]
            rr_prev[i] = initial_mean_rr
            rr_local_mean[i] = np.mean([initial_mean_rr, rr_ms_all[0]])
        else:
            rr_current[i] = rr_ms_all[i - 1]
            rr_prev[i] = rr_ms_all[i - 2]
            
            # Local moving average of the last 5 beats (beats i-4 to i)
            window_intervals = rr_ms_all[max(0, i - 4):i]
            rr_local_mean[i] = np.mean(window_intervals)
            
        rr_ratio[i] = rr_current[i] / rr_local_mean[i]

    # --- B. Morphological and Repolarization Features ---
    r_amplitude = np.zeros(n_beats)
    qrs_duration = np.zeros(n_beats)
    qrs_energy = np.zeros(n_beats)
    st_mean = np.zeros(n_beats)
    
    for i, r_idx in enumerate(r_peaks):
        # 5. R_amplitude: Signal amplitude at the R peak
        r_amplitude[i] = signal[r_idx]
        
       # 6. QRS_duration: Width of the QRS complex above 50% of the R-peak amplitude
        threshold = 0.5 * r_amplitude[i]
        
        # Search backwards (to the left)
        left_idx = r_idx
        while left_idx > 0 and signal[left_idx] > threshold:
            left_idx -= 1
            if (r_idx - left_idx) > 100:  # Safety threshold against noise
                break
                
       # Search forward (to the right)
        right_idx = r_idx
        while right_idx < len(signal) - 1 and signal[right_idx] > threshold:
            right_idx += 1
            if (right_idx - r_idx) > 100:
                break
                
        qrs_duration[i] = right_idx - left_idx
        
        # 7. QRS_energy: Sum of squared values within a ±20-sample window
        qrs_start = max(0, r_idx - 20)
        qrs_end = min(len(signal), r_idx + 21)
        qrs_energy[i] = np.sum(signal[qrs_start:qrs_end] ** 2)
        
        # 8. ST_mean: Mean signal value in the ST segment (+40 to +120 samples post-R)
        st_start = min(len(signal), r_idx + 40)
        st_end = min(len(signal), r_idx + 121)
        if st_start < st_end:
            st_mean[i] = np.mean(signal[st_start:st_end])
        else:
            st_mean[i] = 0.0

    # Create a copy of the DataFrame for this record and assign the variables
    df_res = df_rec_beats.copy()
    df_res["RR_current"] = rr_current
    df_res["RR_prev"] = rr_prev
    df_res["RR_ratio"] = rr_ratio
    df_res["RR_local_mean"] = rr_local_mean
    df_res["R_amplitude"] = r_amplitude
    df_res["QRS_duration"] = qrs_duration
    df_res["QRS_energy"] = qrs_energy
    df_res["ST_mean"] = st_mean
    
    return df_res

## Main processing loop

In [4]:
extracted_features_list = []

for rec_id in records_to_process:
    df_rec_beats = mapped_beats_df[mapped_beats_df["record_id"] == rec_id]
    df_feat = extract_features_for_record(rec_id, df_rec_beats)
    extracted_features_list.append(df_feat)

# Concatenate all records in a single unified DataFrame 
final_features_df = pd.concat(extracted_features_list, ignore_index=True)

-> Processing record 100...
-> Processing record 101...
-> Processing record 105...
-> Processing record 106...
-> Processing record 108...
-> Processing record 109...
-> Processing record 111...
-> Processing record 112...
-> Processing record 115...
-> Processing record 117...
-> Processing record 119...
-> Processing record 201...
-> Processing record 213...
-> Processing record 219...


In [5]:
final_features_df.head()

,record_id,sample,raw_symbol,label,RR_current,RR_prev,RR_ratio,RR_local_mean,R_amplitude,QRS_duration,QRS_energy,ST_mean
0,100,77,N,N,798.888889,798.888889,1.000000,798.888889,0.840,6.0,6.79495,-0.342469
1,100,370,N,N,813.888889,798.888889,1.009301,806.388889,0.940,8.0,8.95210,-0.409691
2,100,662,N,N,811.111111,813.888889,0.998291,812.500000,0.885,6.0,9.14830,-0.393580
3,100,946,N,N,788.888889,811.111111,0.980437,804.629630,0.810,6.0,9.52190,-0.364815
4,100,1231,N,N,791.666667,788.888889,0.987868,801.388889,0.820,6.0,8.07610,-0.403025


## Saving the results in the "2_extracted" folder

In [6]:
os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
final_features_df.to_csv(PATH_OUTPUT, index=False)

print("\n=====================================================================")
print("Process completed successfully!")
print(f"Feature matrix saved at: {PATH_OUTPUT}")
print(f"Shape of generated file: {final_features_df.shape}")
print("=====================================================================")


Process completed successfully!
Feature matrix saved at: ../data/2_extracted/features_extracted.csv
Shape of generated file: (30162, 12)


## <span style="color:red">REMEMBER!! Retain a 'record_id' column for traceability but do not include it as a model feature.</span>